## NC Baseline — Fashion-MNIST

Replicates the MNIST two-phase NC experiment on Fashion-MNIST (Xiao et al., 2017).
Identical pipeline: MLP-5, ReLU, λ=1e-4, Adam, cosine LR, CE→MSE two-phase protocol.
3 seeds. Expected runtime: ~30 min on Colab T4.

Key question: does fn* concentrate at a similar value to MNIST (≈1.052),
or does the more complex task shift it — extending the (model, dataset) grid?

In [1]:
import torch, torchvision, time
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
import torchvision.transforms as T

torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE_DIR = '/content/'
assert torch.cuda.is_available(), 'No GPU — Runtime → Change runtime type → T4'
print(f'GPU: {torch.cuda.get_device_name(0)}  |  PyTorch: {torch.__version__}')

# Known thresholds from paper (for comparison)
MNIST_MLP5_FN    = 1.052   # ReLU MLP-5 / MNIST
CIFAR_RESNET_FN  = 1.515   # ReLU ResNet-20 / CIFAR-10
print(f'Reference fn* — MNIST MLP-5: {MNIST_MLP5_FN}  |  CIFAR ResNet-20: {CIFAR_RESNET_FN}')


GPU: NVIDIA A100-SXM4-40GB  |  PyTorch: 2.10.0+cu128
Reference fn* — MNIST MLP-5: 1.052  |  CIFAR ResNet-20: 1.515


In [2]:
# Fashion-MNIST — same format as MNIST (28x28, grayscale, 10 classes)
# Mean/std from dataset statistics
transform = T.Compose([T.ToTensor(), T.Normalize((0.2860,), (0.3530,))])

trainset = torchvision.datasets.FashionMNIST('/content/data', train=True,
                                              download=True, transform=transform)
testset  = torchvision.datasets.FashionMNIST('/content/data', train=False,
                                              download=True, transform=transform)

train_loader = DataLoader(trainset, batch_size=512, shuffle=True,
                          num_workers=4, persistent_workers=True,
                          prefetch_factor=2, pin_memory=True)
test_loader  = DataLoader(testset,  batch_size=1024, shuffle=False,
                          num_workers=4, persistent_workers=True,
                          prefetch_factor=2, pin_memory=True)

classes = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
           'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Boot']
print(f'Fashion-MNIST: {len(trainset):,} train / {len(testset):,} test')
print(f'Classes: {classes}')


100%|██████████| 26.4M/26.4M [00:02<00:00, 10.8MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 172kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.31MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 15.5MB/s]

Fashion-MNIST: 60,000 train / 10,000 test
Classes: ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Boot']


In [3]:
class MLP(nn.Module):
    def __init__(self, depth=5, width=512, act_cls=nn.ReLU, num_classes=10):
        super().__init__()
        layers = [nn.Flatten(), nn.Linear(784, width), act_cls()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), act_cls()]
        self.body = nn.Sequential(*layers)
        self.head = nn.Linear(width, num_classes)
        self._feats = None
        self.body.register_forward_hook(
            lambda m, i, o: setattr(self, '_feats', o.detach()))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)
    def forward(self, x): return self.head(self.body(x))
    def get_features(self, x): self(x); return self._feats
    def get_classifier_weights(self): return self.head.weight.detach()

m = MLP()
print(f'MLP-5 width=512: {sum(p.numel() for p in m.parameters())/1e6:.2f}M params')
del m


MLP-5 width=512: 1.46M params


In [4]:
@torch.no_grad()
def compute_nc(model, loader, K=10):
    model.eval()
    fl, ll = [], []
    for x, y in loader:
        fl.append(model.get_features(x.to(DEVICE, non_blocking=True)))
        ll.append(y.to(DEVICE, non_blocking=True))
    H = torch.cat(fl); Y = torch.cat(ll)
    mu_G = H.mean(0)
    mu_c = torch.stack([H[Y==c].mean(0) for c in range(K)])
    M    = mu_c - mu_G
    Sw   = sum((H[Y==c]-mu_c[c]).T@(H[Y==c]-mu_c[c]) for c in range(K))/len(H)
    Sb   = M.T @ M / K
    nc1  = (torch.trace(Sw)/torch.trace(Sb).clamp(1e-10)).item()
    Mn   = F.normalize(M, dim=1)
    cos  = Mn @ Mn.T
    mask = ~torch.eye(K, dtype=torch.bool, device=DEVICE)
    nc2  = (cos[mask]-(-1./(K-1))).abs().mean().item()
    Wn   = F.normalize(model.get_classifier_weights().to(DEVICE), dim=1)
    nc3  = (1-(Mn*Wn).sum(1).mean()).item()
    return {'nc1':nc1, 'nc2':nc2, 'nc3':nc3,
            'feat_norm':H.norm(dim=1).mean().item()}

def evaluate(model, loader):
    model.eval(); correct=total=0
    with torch.no_grad():
        for x,y in loader:
            x,y=x.to(DEVICE,non_blocking=True),y.to(DEVICE,non_blocking=True)
            correct+=(model(x).argmax(1)==y).sum().item(); total+=len(y)
    return correct/total

print('NC metrics and evaluate ready.')


NC metrics and evaluate ready.


In [5]:
def run_twophase(model, name, lr=1e-3, wd=1e-4,
                 phase1=200, phase2=500, nc_every=10, nc_thresh=0.01):
    try:
        model = torch.compile(model, mode='reduce-overhead')
    except Exception:
        pass
    model = model.to(DEVICE)
    K=10; rows=[]; terminal=False; t_nc=None; t0=time.time()

    for phase, loss_fn, n_ep in [(1,'ce',phase1),(2,'mse',phase2)]:
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_ep)
        off = phase1 if phase==2 else 0

        for ep_l in range(1, n_ep+1):
            ep = off + ep_l
            model.train()
            for x,y in train_loader:
                x,y=x.to(DEVICE,non_blocking=True),y.to(DEVICE,non_blocking=True)
                opt.zero_grad(set_to_none=True)
                logits = model(x)
                loss = (F.mse_loss(logits, F.one_hot(y,K).float())
                        if loss_fn=='mse' else F.cross_entropy(logits,y))
                loss.backward(); opt.step()
            sch.step()

            if ep_l % nc_every == 0 or ep_l == n_ep:
                tr = evaluate(model, train_loader)
                te = evaluate(model, test_loader)
                if tr >= 0.99 and not terminal:
                    terminal = True
                    print(f'  [{name}] Terminal phase at epoch {ep}')
                nc = compute_nc(model, train_loader) if terminal else                      {'nc1':None,'nc2':None,'nc3':None,'feat_norm':None}
                rows.append({'epoch':ep,'phase':phase,'train':tr,'test':te,**nc})
                if t_nc is None and nc.get('nc1') and nc['nc1'] < nc_thresh:
                    t_nc = ep
                    fn   = nc['feat_norm']
                    print(f'  [{name}] NC1 collapsed at epoch {t_nc}  fn*={fn:.4f}')

    elapsed = time.time()-t0
    fn_at_tnc = next((r['feat_norm'] for r in rows if r['epoch']==t_nc), None) if t_nc else None
    print(f'  [{name}] Done in {elapsed/60:.1f} min  T_NC={t_nc}  fn*={fn_at_tnc}')
    return pd.DataFrame(rows), t_nc, fn_at_tnc

print('run_twophase ready.')


run_twophase ready.


In [6]:
# ── Main experiment: 3 seeds, MLP-5, ReLU, wd=1e-4 ──────────────────────
results = []

for seed in range(3):
    print(f'\n=== Fashion-MNIST  seed={seed} ===')
    torch.manual_seed(seed)
    model = MLP(depth=5, width=512, act_cls=nn.ReLU)
    df, t_nc, fn = run_twophase(model, f'fmnist-s{seed}',
                                 lr=1e-3, wd=1e-4,
                                 phase1=200, phase2=500)
    df.to_csv(f'{SAVE_DIR}fmnist_s{seed}.csv', index=False)
    results.append({'seed':seed, 'T_NC':t_nc, 'fn':fn,
                    'test_acc': df.test.iloc[-1]})

df_summary = pd.DataFrame(results)
df_summary.to_csv(f'{SAVE_DIR}fmnist_summary.csv', index=False)

print('\n=== FASHION-MNIST SUMMARY ===')
print(df_summary.to_string(index=False))

confirmed = df_summary.dropna(subset=['fn'])
if len(confirmed) > 1:
    m, s = confirmed.fn.mean(), confirmed.fn.std()
    cv   = s / m * 100
    print(f'\nfn*: mean={m:.4f}  std={s:.4f}  CV={cv:.1f}%  N={len(confirmed)}')
    print(f'\nComparison:')
    print(f'  MNIST      MLP-5  fn* = {MNIST_MLP5_FN:.3f}')
    print(f'  FashionMNIST MLP-5 fn* = {m:.3f}  ({"higher" if m > MNIST_MLP5_FN else "lower"} by {abs(m-MNIST_MLP5_FN)/MNIST_MLP5_FN*100:.1f}%)')
    print(f'  CIFAR-10   ResNet  fn* = {CIFAR_RESNET_FN:.3f}')



=== Fashion-MNIST  seed=0 ===
  [fmnist-s0] Terminal phase at epoch 70
  [fmnist-s0] Done in 42.6 min  T_NC=None  fn*=None

=== Fashion-MNIST  seed=1 ===
  [fmnist-s1] Terminal phase at epoch 50
  [fmnist-s1] NC1 collapsed at epoch 590  fn*=1.1087
  [fmnist-s1] Done in 42.5 min  T_NC=590  fn*=1.1087390184402466

=== Fashion-MNIST  seed=2 ===
  [fmnist-s2] Terminal phase at epoch 60
  [fmnist-s2] Done in 42.4 min  T_NC=None  fn*=None

=== FASHION-MNIST SUMMARY ===
 seed  T_NC       fn  test_acc
    0   NaN      NaN    0.8947
    1 590.0 1.108739    0.8952
    2   NaN      NaN    0.8902
